In [ ]:
# Deep Q-Network (DQN)
#
# https://docs.pytorch.org/rl/stable/tutorials/getting-started-5.html

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torch import nn
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import GymEnv, StepCounter, TransformedEnv
from torchrl.modules import EGreedyModule, QValueActor
from torchrl.objectives import DQNLoss, SoftUpdate, ValueEstimators
from torchrl import logger

_ = torch.manual_seed(0)

In [ ]:
# CartPole has 2 (discrete) one-hot encoded actions
env = TransformedEnv(GymEnv("CartPole-v1"), StepCounter())
_ = env.set_seed(0)

In [4]:
# Value network used to estimate the action-value function Q(s,a).
# The network maps a state s to the vector of action-values [Q(s,a) : a action].
value_net = nn.Sequential(
    nn.LazyLinear(64),
    nn.Tanh(),
    nn.LazyLinear(64),
    nn.Tanh(),
    nn.LazyLinear(env.action_spec.shape[-1]),
)

# wrap value network to work with tensordict as input/output
value_module = TensorDictModule(
    value_net, in_keys=["observation"], out_keys=["action_value"]
)

# deterministic policy which selects the action with the highest action value
# state s -> action = argmax_a Q(s,a)
policy_module = QValueActor(value_module, spec=env.action_spec)

_ = env.rollout(10, policy_module)  # initialize lazy layers

In [ ]:
# module to compute the loss: L = Q(s,a) - (r + gamma * max_a Q(s',a))
loss_module = DQNLoss(value_network=policy_module, action_space=env.action_spec)
loss_module.make_value_estimator(ValueEstimators.TD0, gamma=0.99)  # default value

updater = SoftUpdate(loss_module, eps=0.99)  # soft-updater for target weights

optimizer = torch.optim.Adam(loss_module.parameters(), lr=0.02)

In [7]:
# create policy with added noise for initial exploration

exploration_module = EGreedyModule(
    spec=env.action_spec,
    eps_init=0.5,
    annealing_num_steps=100_000,
)

policy_explore = TensorDictSequential([policy_module, exploration_module])

In [ ]:
INIT_RANDOM_FRAMES = 5000
FRAMES_PER_BATCH = 100

collector = Collector(
    env,
    policy=policy_explore,
    frames_per_batch=FRAMES_PER_BATCH,
    init_random_frames=INIT_RANDOM_FRAMES,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(100_000))

In [ ]:
OPTIM_STEPS = 10
BATCH_SIZE = 128

step_count = 0
episode_count = 0


for idx, data in enumerate(collector):
    buffer.extend(data)  # add data to replay buffer

    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    if len(buffer) < collector.init_random_frames:
        continue

    # length of longest trajectory in batch
    max_steps = data["next", "step_count"].max()

    # define the stopping condition as reaching 200 steps
    if max_steps > 200:
        break

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] max steps: {max_steps:>3}")

    for _ in range(OPTIM_STEPS):
        sample = buffer.sample(BATCH_SIZE)

        loss_module(sample)["loss"].backward()

        optimizer.step()
        optimizer.zero_grad()

        updater.step()  # update target params

    exploration_module.step(data.numel())  # update exploration factor

logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-14 15:14:24,606 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
2026-02-14 15:14:28,684 [torchrl][INFO]    [50] max steps: 26 [END]
2026-02-14 15:14:30,001 [torchrl][INFO]    [60] max steps: 59 [END]
2026-02-14 15:14:31,318 [torchrl][INFO]    [70] max steps: 104 [END]
2026-02-14 15:14:31,579 [torchrl][INFO]    solved after 2400 steps, 74 episodes [END]
